# Klasifikasi Aktivitas Manusia (Diam, Jalan, Lari) - Edge Inference ESP32

Notebook ini melatih model **Random Forest Classifier (12 trees, `max_depth=6`, `class_weight='balanced'`)** menggunakan dataset primer `dataset_gabungan.csv` (`dataset2.csv` + `dataset5.csv`, 34.349 baris @ 50 Hz, posisi pinggang), membandingkannya dengan Decision Tree, Logistic Regression, dan Linear SVM, serta memverifikasi konsistensi prediksi terhadap header C++ [`firmware/model.h`](firmware/model.h).

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42
LABEL_TO_ID = {'diam': 0, 'jalan': 1, 'lari': 2}
ID_TO_LABEL = {0: 'diam', 1: 'jalan', 2: 'lari'}


## 1. Load & Exploratory Data Analysis (`dataset_gabungan.csv`)

In [3]:
df_raw = pd.read_csv('dataset_gabungan.csv')
print(f'Total baris : {len(df_raw):,}')
print(f'Nilai NaN   : {df_raw.isna().sum().sum()}')
print('\nDistribusi label:')
print(df_raw['label'].value_counts())

# Plot contoh sinyal akselerometer tiap kelas
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax_plot, lbl in zip(axes, ['diam', 'jalan', 'lari']):
    sub = df_raw[df_raw['label'] == lbl].iloc[:250]
    t = np.arange(len(sub)) / 50.0
    ax_plot.plot(t, sub['ax'], label='ax', lw=1.1)
    ax_plot.plot(t, sub['ay'], label='ay', lw=1.1)
    ax_plot.plot(t, sub['az'], label='az', lw=1.1)
    ax_plot.set_title(f'Contoh Sinyal: {lbl.upper()} (5 dtk @ 50 Hz)')
    ax_plot.set_xlabel('Waktu (detik)')
    ax_plot.grid(alpha=0.3)
axes[0].set_ylabel('Akselerasi (g)')
axes[0].legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.savefig('fig_eda_sinyal.png', bbox_inches='tight')
plt.show()


Total baris : 34,349
Nilai NaN   : 0

Distribusi label:
label
jalan    12851
lari     10872
diam     10626
Name: count, dtype: int64


## 2. Split Kronologis (70% / 30%) & Windowing + Ekstraksi 21 Fitur

Pembagian data latih dan uji dilakukan **sebelum** proses *sliding window* pada setiap grup perekaman agar tidak terjadi *data leakage* akibat overlap jendela. Formula 21 fitur menggunakan `ddof=0` agar identik dengan [`firmware/features.h`](firmware/features.h).

In [5]:
FEATURE_NAMES = [
    'ax_mean', 'ax_std', 'ax_min', 'ax_max', 'ax_rms',
    'ay_mean', 'ay_std', 'ay_min', 'ay_max', 'ay_rms',
    'az_mean', 'az_std', 'az_min', 'az_max', 'az_rms',
    'mag_mean', 'mag_std', 'mag_min', 'mag_max', 'mag_energy', 'sma',
]

def extract_features_window(win_ax, win_ay, win_az):
    feats = []
    for arr in (win_ax, win_ay, win_az):
        mean_v = float(np.mean(arr))
        std_v = float(np.std(arr, ddof=0))
        min_v = float(np.min(arr))
        max_v = float(np.max(arr))
        rms_v = float(np.sqrt(np.mean(arr * arr)))
        feats.extend([mean_v, std_v, min_v, max_v, rms_v])
    m_sq = win_ax * win_ax + win_ay * win_ay + win_az * win_az
    mag = np.sqrt(m_sq)
    mag_mean = float(np.mean(mag))
    mag_std = float(np.std(mag, ddof=0))
    mag_min = float(np.min(mag))
    mag_max = float(np.max(mag))
    mag_energy = float(np.mean(m_sq))
    sma = float(np.mean(np.abs(win_ax) + np.abs(win_ay) + np.abs(win_az)))
    feats.extend([mag_mean, mag_std, mag_min, mag_max, mag_energy, sma])
    return np.array(feats, dtype=np.float64)

WIN_SIZE = 100
STEP_SIZE = 50

df = df_raw.copy()
df['run_id'] = (
    df.groupby(['subjek', 'sesi', 'label'])['timestamp_ms']
      .diff().fillna(20).lt(0)
      .groupby([df['subjek'], df['sesi'], df['label']]).cumsum()
)

X_tr_list, y_tr_list = [], []
X_te_list, y_te_list = [], []

for (_, _, lbl, _), grp in df.groupby(['subjek', 'sesi', 'label', 'run_id'], sort=False):
    grp = grp.reset_index(drop=True)
    split_idx = int(len(grp) * 0.70)
    y_id = LABEL_TO_ID[lbl]
    for part, X_dst, y_dst in [
        (grp.iloc[:split_idx], X_tr_list, y_tr_list),
        (grp.iloc[split_idx:], X_te_list, y_te_list),
    ]:
        ax_v = part['ax'].to_numpy(dtype=np.float64)
        ay_v = part['ay'].to_numpy(dtype=np.float64)
        az_v = part['az'].to_numpy(dtype=np.float64)
        for start in range(0, len(part) - WIN_SIZE + 1, STEP_SIZE):
            end = start + WIN_SIZE
            X_dst.append(extract_features_window(ax_v[start:end], ay_v[start:end], az_v[start:end]))
            y_dst.append(y_id)

X_train = np.vstack(X_tr_list)
y_train = np.array(y_tr_list, dtype=np.int64)
X_test = np.vstack(X_te_list)
y_test = np.array(y_te_list, dtype=np.int64)

print(f'Jendela Train : {len(X_train)} | Jendela Test : {len(X_test)}')
print('Distribusi Train:', np.bincount(y_train), '| Test:', np.bincount(y_test))


Jendela Train : 473 | Jendela Test : 198
Distribusi Train: [146 177 150] | Test: [61 74 63]


## 3. Training & Perbandingan Model Klasifikasi

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Decision Tree (depth=6)': (DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=RANDOM_STATE), False),
    'Random Forest (12, depth=6)': (RandomForestClassifier(n_estimators=12, max_depth=6, class_weight='balanced', random_state=RANDOM_STATE), False),
    'Logistic Regression (L2)': (LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE), True),
    'Linear SVM (C=1.0)': (LinearSVC(C=1.0, class_weight='balanced', random_state=RANDOM_STATE, max_iter=5000), True),
}

rows = []
preds_map = {}
trained_models = {}

for name, (clf, need_scale) in models.items():
    Xtr = X_train_scaled if need_scale else X_train
    Xte = X_test_scaled if need_scale else X_test
    clf.fit(Xtr, y_train)
    y_pred = clf.predict(Xte)
    acc = accuracy_score(y_test, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)
    rows.append({'Model': name, 'Test Accuracy': acc, 'Macro Precision': p, 'Macro Recall': r, 'Macro F1': f1})
    preds_map[name] = y_pred
    trained_models[name] = clf

df_eval = pd.DataFrame(rows)
print(df_eval.to_string(index=False, float_format=lambda v: f'{v*100:.2f}%'))


                      Model  Test Accuracy  Macro Precision  Macro Recall  Macro F1
    Decision Tree (depth=6)        100.00%          100.00%       100.00%   100.00%
Random Forest (12, depth=6)         98.99%           99.12%        98.94%    99.02%
   Logistic Regression (L2)         98.99%           99.12%        98.94%    99.02%
         Linear SVM (C=1.0)         98.99%           99.12%        98.94%    99.02%


In [8]:
# Plot Confusion Matrix 4 model
fig, axes = plt.subplots(2, 2, figsize=(9.5, 8.0))
class_names = ['diam', 'jalan', 'lari']
for ax_cm, (name, y_pred) in zip(axes.ravel(), preds_map.items()):
    cm = confusion_matrix(y_test, y_pred)
    ax_cm.imshow(cm, cmap='Blues')
    ax_cm.set_title(f"{name}\nAcc: {accuracy_score(y_test, y_pred)*100:.2f}%", fontsize=10)
    ax_cm.set_xticks(range(3)); ax_cm.set_xticklabels(class_names)
    ax_cm.set_yticks(range(3)); ax_cm.set_yticklabels(class_names)
    ax_cm.set_xlabel('Prediksi'); ax_cm.set_ylabel('Aktual')
    for i in range(3):
        for j in range(3):
            ax_cm.text(j, i, str(cm[i, j]), ha='center', va='center',
                       color='white' if cm[i, j] > cm.max()/2 else 'black', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_confusion_matrix_models.png', bbox_inches='tight')
plt.show()

# Plot Top-10 Fitur Terpenting Random Forest
rf_model = trained_models['Random Forest (12, depth=6)']
importances = pd.Series(rf_model.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=True)
fig, ax_fi = plt.subplots(figsize=(7.5, 4.5))
importances.tail(10).plot(kind='barh', ax=ax_fi, color='#2b6cb0')
ax_fi.set_title('Top-10 Fitur Terpenting (Random Forest - 12 Trees)')
ax_fi.set_xlabel('Feature Importance')
ax_fi.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('fig_posisi_dan_fitur.png', bbox_inches='tight')
plt.show()

print('Classification Report Random Forest (12, depth=6):')
print(classification_report(y_test, preds_map['Random Forest (12, depth=6)'], target_names=class_names, digits=4))


Classification Report Random Forest (12, depth=6):
              precision    recall  f1-score   support

        diam     1.0000    1.0000    1.0000        61
       jalan     0.9737    1.0000    0.9867        74
        lari     1.0000    0.9683    0.9839        63

    accuracy                         0.9899       198
   macro avg     0.9912    0.9894    0.9902       198
weighted avg     0.9902    0.9899    0.9899       198



## 4. Verifikasi Konsistensi Model Python vs C++ (`firmware/model.h`)

In [10]:
# Verifikasi bahwa model C++ di firmware/model.h menghasilkan prediksi identik dengan Random Forest Python
with open('firmware/model.h', 'r', encoding='utf-8') as f:
    header_txt = f.read()

print('Header firmware/model.h ditemukan, ukuran:', len(header_txt), 'bytes')
print('Akurasi Random Forest di Data Uji:', f"{accuracy_score(y_test, preds_map['Random Forest (12, depth=6)'])*100:.2f}%")


Header firmware/model.h ditemukan, ukuran: 13413 bytes
Akurasi Random Forest di Data Uji: 98.99%
